# 33. 統計学から機械学習へ：橋渡しノートブック

## 概要

このノートブックでは、**統計学の概念**が**機械学習アルゴリズム**にどのように対応するかを解説します。
statspgで学んだ統計理論が、mlpgで扱う実践的なMLにどうつながるかを理解することで、両分野の理解を深めます。

## 対応関係の全体像

| 統計学の概念 | 機械学習の概念 |
|---|---|
| 線形回帰（最小二乗法） | 線形モデル（回帰） |
| 最尤推定（MLE） | 損失関数最小化 |
| ベイズ推定（MAP推定） | 正則化（Ridge/LASSO） |
| 仮説検定・p値 | モデル評価指標・交差検証 |
| 情報量基準（AIC/BIC） | モデル選択・過学習防止 |

## 目次
1. 線形回帰 → 線形モデルの対応
2. 最尤推定 → 損失関数最小化
3. ベイズ推定 → 正則化（Ridge/LASSO）
4. 仮説検定 → モデル評価指標
5. 情報量基準 → モデル選択
6. まとめ：統計的視点でMLを読む

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import scipy.stats as stats
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.datasets import make_regression, make_classification
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# 再現性のためのシード固定
np.random.seed(42)

# プロット設定
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('ライブラリの読み込み完了')

---
## 1. 線形回帰（統計） → 線形モデル（ML）

### 統計学の視点
統計学では線形回帰を以下のように定義します：

$$y = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p + \varepsilon, \quad \varepsilon \sim N(0, \sigma^2)$$

**最小二乗推定量（OLS）**:
$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

### 機械学習の視点
MLでは同じ問題を「損失関数の最小化」として捉えます：

$$\min_{\mathbf{w}} \mathcal{L}(\mathbf{w}) = \min_{\mathbf{w}} \frac{1}{n} \sum_{i=1}^{n} (y_i - \mathbf{w}^\top \mathbf{x}_i)^2$$

**本質は同じ**: OLSの最小二乗法 ＝ MSE損失の最小化

In [ ]:
# データ生成
n = 100
X = np.random.randn(n, 2)
true_beta = np.array([3.0, -1.5])
y = X @ true_beta + 2.0 + np.random.randn(n) * 0.5  # ノイズあり

# --- 統計学の手法：OLS（解析的解） ---
X_with_intercept = np.column_stack([np.ones(n), X])
beta_ols = np.linalg.inv(X_with_intercept.T @ X_with_intercept) @ X_with_intercept.T @ y

# --- 機械学習の手法：scikit-learn LinearRegression ---
lr = LinearRegression()
lr.fit(X, y)

print('=== 統計学 vs 機械学習：線形回帰の係数比較 ===')
print(f'\n真の係数: β0=2.0, β1={true_beta[0]}, β2={true_beta[1]}')
print(f'\n[統計 OLS] 切片={beta_ols[0]:.4f}, β1={beta_ols[1]:.4f}, β2={beta_ols[2]:.4f}')
print(f'[ML sklearn] 切片={lr.intercept_:.4f}, β1={lr.coef_[0]:.4f}, β2={lr.coef_[1]:.4f}')
print('\n→ 全く同じ結果！（OLS = MSE最小化）')

# 残差分析（統計）vs 予測誤差（ML）
y_pred_ols = X_with_intercept @ beta_ols
y_pred_ml = lr.predict(X)

print(f'\n=== 評価指標の対応 ===')
print(f'統計: 残差平方和 (RSS) = {np.sum((y - y_pred_ols)**2):.4f}')
print(f'ML:   MSE × n       = {mean_squared_error(y, y_pred_ml) * n:.4f}')
print(f'統計: 決定係数 R²   = {r2_score(y, y_pred_ols):.4f}')
print(f'ML:   R² スコア     = {r2_score(y, y_pred_ml):.4f}')

In [ ]:
# 視覚化：残差プロット（統計的診断 ≒ MLの誤差分析）
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 統計：残差の正規性確認
residuals = y - y_pred_ols
axes[0].scatter(y_pred_ols, residuals, alpha=0.6, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('予測値 (fitted values)')
axes[0].set_ylabel('残差 (residuals)')
axes[0].set_title('[統計] 残差プロット\n（等分散性・無相関性の確認）')

# ML：予測 vs 実測
axes[1].scatter(y, y_pred_ml, alpha=0.6, color='coral')
axes[1].plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2)
axes[1].set_xlabel('実測値 (true values)')
axes[1].set_ylabel('予測値 (predicted values)')
axes[1].set_title(f'[ML] 予測 vs 実測\n（R² = {r2_score(y, y_pred_ml):.3f}）')

plt.suptitle('線形回帰：統計的診断 ≈ MLの予測評価', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('💡 ポイント: 統計の「残差の等分散性確認」= MLの「予測誤差の分布確認」')

---
## 2. 最尤推定（MLE） → 損失関数最小化

### 統計学の視点：最尤推定
観測データ $\mathbf{y} = (y_1, \ldots, y_n)$ が正規分布 $N(\mathbf{X}\boldsymbol{\beta}, \sigma^2 \mathbf{I})$ に従うとすると、
対数尤度は：

$$\ell(\boldsymbol{\beta}) = -\frac{n}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\|\mathbf{y} - \mathbf{X}\boldsymbol{\beta}\|^2$$

これを最大化 → $\|\mathbf{y} - \mathbf{X}\boldsymbol{\beta}\|^2$ を**最小化**することと同値！

### 機械学習の視点：損失関数
- **回帰（正規分布仮定）**: MSE損失 = 負の対数尤度
- **分類（ベルヌーイ分布仮定）**: 交差エントロピー損失 = 負の対数尤度

$$\text{Cross-Entropy} = -\sum_{i} y_i \log p_i + (1-y_i)\log(1-p_i)$$

In [ ]:
# MLE と損失関数の等価性を示す
# --- ロジスティック回帰の例 ---

# 分類データ生成
X_cls, y_cls = make_classification(n_samples=200, n_features=2, n_informative=2,
                                    n_redundant=0, random_state=42)

# sklearn のロジスティック回帰（内部で交差エントロピーを最小化 = MLE）
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_cls, y_cls)

# MLE 手動計算（シグモイド関数 + 対数尤度）
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_likelihood(X, y, w, b):
    z = X @ w + b
    p = sigmoid(z)
    # 数値安定性のためクリッピング
    p = np.clip(p, 1e-10, 1-1e-10)
    return np.sum(y * np.log(p) + (1-y) * np.log(1-p))

ll = log_likelihood(X_cls, y_cls, log_reg.coef_[0], log_reg.intercept_[0])
y_pred_cls = log_reg.predict(X_cls)

print('=== 最尤推定 vs 交差エントロピー損失最小化 ===')
print(f'\nロジスティック回帰（sklearn）')
print(f'  係数: {log_reg.coef_[0]}')
print(f'  切片: {log_reg.intercept_[0]:.4f}')
print(f'\n対数尤度 ℓ(β) = {ll:.4f}')
print(f'交差エントロピー損失 = {-ll/len(y_cls):.4f}')
print(f'精度 = {accuracy_score(y_cls, y_pred_cls):.4f}')
print('\n→ 交差エントロピー最小化 = 対数尤度最大化（MLE）')

In [ ]:
# 損失関数の分布仮定との対応を視覚化
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

x_range = np.linspace(-4, 4, 200)

# 1. 正規分布 → MSE損失
axes[0].plot(x_range, stats.norm.pdf(x_range, 0, 1), 'b-', lw=2)
axes[0].fill_between(x_range, stats.norm.pdf(x_range, 0, 1), alpha=0.3)
axes[0].set_title('正規分布 N(μ, σ²)\n→ MSE損失')
axes[0].set_xlabel('y - ŷ')
axes[0].set_ylabel('確率密度')
axes[0].text(0.05, 0.95, r'$\mathcal{L}_{MSE} = \sum(y_i - \hat{y}_i)^2$',
             transform=axes[0].transAxes, fontsize=10, va='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 2. ラプラス分布 → MAE損失
axes[1].plot(x_range, stats.laplace.pdf(x_range, 0, 1), 'g-', lw=2)
axes[1].fill_between(x_range, stats.laplace.pdf(x_range, 0, 1), alpha=0.3, color='green')
axes[1].set_title('ラプラス分布 Lap(μ, b)\n→ MAE損失')
axes[1].set_xlabel('y - ŷ')
axes[1].set_ylabel('確率密度')
axes[1].text(0.05, 0.95, r'$\mathcal{L}_{MAE} = \sum|y_i - \hat{y}_i|$',
             transform=axes[1].transAxes, fontsize=10, va='top',
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

# 3. ベルヌーイ分布 → 交差エントロピー
p_range = np.linspace(0.01, 0.99, 200)
ce_loss = -np.log(p_range)  # y=1 の場合
axes[2].plot(p_range, ce_loss, 'r-', lw=2)
axes[2].fill_between(p_range, ce_loss, alpha=0.3, color='red')
axes[2].set_title('ベルヌーイ分布 Ber(p)\n→ 交差エントロピー損失')
axes[2].set_xlabel('予測確率 p')
axes[2].set_ylabel('損失 -log(p)')
axes[2].set_ylim(0, 5)
axes[2].text(0.05, 0.95, r'$\mathcal{L}_{CE} = -\sum y_i \log p_i$',
             transform=axes[2].transAxes, fontsize=10, va='top',
             bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.5))

plt.suptitle('統計の分布仮定 = MLの損失関数選択', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('💡 ポイント: どの損失関数を使うかは、どの確率分布を仮定するかと等価！')

---
## 3. ベイズ推定（MAP） → 正則化（Ridge/LASSO）

### 統計学の視点：MAP推定
ベイズ推定では事後分布を最大化（MAP推定）します：

$$\hat{\boldsymbol{\beta}}_{MAP} = \arg\max_{\boldsymbol{\beta}} p(\boldsymbol{\beta}|\mathbf{y}) = \arg\max_{\boldsymbol{\beta}} \underbrace{p(\mathbf{y}|\boldsymbol{\beta})}_{\text{尤度}} \cdot \underbrace{p(\boldsymbol{\beta})}_{\text{事前分布}}$$

### 事前分布と正則化の対応

| 事前分布 | MAP推定の損失 | MLの正則化 |
|---|---|---|
| $\boldsymbol{\beta} \sim N(0, \tau^2 I)$（正規分布） | MSE + $\lambda\|\boldsymbol{\beta}\|^2$ | **Ridge回帰** (L2正則化) |
| $\boldsymbol{\beta} \sim \text{Laplace}(0, b)$（ラプラス分布） | MSE + $\lambda\|\boldsymbol{\beta}\|_1$ | **LASSO** (L1正則化) |

In [ ]:
# Ridge/LASSOの係数の振る舞いを比較
# 多重共線性のあるデータで正則化の効果を確認

np.random.seed(42)
n_samples, n_features = 100, 20

# 特徴量生成（一部の特徴のみ真に有効）
X_reg, y_reg, true_coef = make_regression(
    n_samples=n_samples, n_features=n_features,
    n_informative=5, noise=10, coef=True, random_state=42
)

# モデル比較
alphas = np.logspace(-3, 3, 50)

ridge_coefs = []
lasso_coefs = []
for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_reg, y_reg)
    ridge_coefs.append(ridge.coef_)

    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_reg, y_reg)
    lasso_coefs.append(lasso.coef_)

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Ridge（L2正則化 = 正規事前分布）
for i in range(n_features):
    axes[0].plot(np.log10(alphas), ridge_coefs[:, i],
                 alpha=0.7, lw=1.5,
                 color='steelblue' if true_coef[i] != 0 else 'lightgray')
axes[0].set_xlabel('log10(λ)  [正則化強度]')
axes[0].set_ylabel('係数の値')
axes[0].set_title('Ridge回帰（L2正則化）\n= 正規事前分布 N(0, τ²) によるMAP推定')
axes[0].axvline(x=0, color='red', linestyle='--', alpha=0.5)
legend_elements = [
    mpatches.Patch(color='steelblue', label='真に有効な特徴'),
    mpatches.Patch(color='lightgray', label='ノイズ特徴')
]
axes[0].legend(handles=legend_elements)
axes[0].text(0.05, 0.05, '係数はゼロに収縮するが\n完全にゼロにはならない',
             transform=axes[0].transAxes, fontsize=9,
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# LASSO（L1正則化 = ラプラス事前分布）
for i in range(n_features):
    axes[1].plot(np.log10(alphas), lasso_coefs[:, i],
                 alpha=0.7, lw=1.5,
                 color='coral' if true_coef[i] != 0 else 'lightgray')
axes[1].set_xlabel('log10(λ)  [正則化強度]')
axes[1].set_ylabel('係数の値')
axes[1].set_title('LASSO（L1正則化）\n= ラプラス事前分布 Lap(0, b) によるMAP推定')
axes[1].axvline(x=0, color='red', linestyle='--', alpha=0.5)
legend_elements2 = [
    mpatches.Patch(color='coral', label='真に有効な特徴'),
    mpatches.Patch(color='lightgray', label='ノイズ特徴')
]
axes[1].legend(handles=legend_elements2)
axes[1].text(0.05, 0.05, '係数が完全にゼロになる\n（スパース解：特徴選択効果）',
             transform=axes[1].transAxes, fontsize=9,
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('ベイズ事前分布 = 正則化：事前分布の形状が疎密を決める', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 事前分布の形状と正則化の幾何学的理解
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 等高線プロット用グリッド
b1_range = np.linspace(-3, 3, 200)
b2_range = np.linspace(-3, 3, 200)
B1, B2 = np.meshgrid(b1_range, b2_range)

# 仮の損失関数（楕円）
loss = (B1 - 2)**2 + 2*(B2 - 0.5)**2

# Ridge制約 (L2球: β1² + β2² ≤ t²)
L2 = B1**2 + B2**2

# LASSO制約 (L1ダイヤモンド: |β1| + |β2| ≤ t)
L1 = np.abs(B1) + np.abs(B2)

# Ridge
cs1 = axes[0].contour(B1, B2, loss, levels=15, cmap='Blues', alpha=0.7)
axes[0].contour(B1, B2, L2, levels=[1.5], colors='red', linewidths=2)
axes[0].fill_between([-3, 3], [-3, -3], [3, 3], where=(L2.min(axis=0) <= 1.5),
                     alpha=0)
theta = np.linspace(0, 2*np.pi, 100)
r = np.sqrt(1.5)
axes[0].plot(r*np.cos(theta), r*np.sin(theta), 'r-', lw=2, label=f'L2制約 (半径={r:.2f})')
axes[0].scatter([2], [0.5], color='blue', s=100, zorder=5, label='制約なし最小点')
axes[0].set_xlim(-3, 3); axes[0].set_ylim(-3, 3)
axes[0].set_xlabel('β₁'); axes[0].set_ylabel('β₂')
axes[0].set_title('Ridge（L2正則化）= 正規事前分布\n→ 円形制約：係数を均等に縮小')
axes[0].legend(fontsize=9)
axes[0].axhline(0, color='k', lw=0.5); axes[0].axvline(0, color='k', lw=0.5)

# LASSO
cs2 = axes[1].contour(B1, B2, loss, levels=15, cmap='Oranges', alpha=0.7)
# L1ダイヤモンド（菱形）
t = 1.5
diamond_x = [t, 0, -t, 0, t]
diamond_y = [0, t, 0, -t, 0]
axes[1].plot(diamond_x, diamond_y, 'r-', lw=2, label=f'L1制約 (t={t})')
axes[1].scatter([2], [0.5], color='blue', s=100, zorder=5, label='制約なし最小点')
axes[1].scatter([0], [t], color='red', s=150, marker='*', zorder=6,
                label='頂点でゼロ交差→スパース解')
axes[1].set_xlim(-3, 3); axes[1].set_ylim(-3, 3)
axes[1].set_xlabel('β₁'); axes[1].set_ylabel('β₂')
axes[1].set_title('LASSO（L1正則化）= ラプラス事前分布\n→ 菱形制約：係数が角で0になりやすい')
axes[1].legend(fontsize=9)
axes[1].axhline(0, color='k', lw=0.5); axes[1].axvline(0, color='k', lw=0.5)

plt.suptitle('正則化の幾何学的理解：制約の形状が解の性質を決める', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('💡 ポイント:')
print('  Ridge（L2）= 正規事前分布 → 係数を均等に縮小（スパースにはならない）')
print('  LASSO（L1）= ラプラス事前分布 → 係数を完全にゼロ化（特徴選択）')

---
## 4. 仮説検定 → モデル評価指標との対応

### 統計学の視点：仮説検定
- **帰無仮説 H₀**: モデルの係数はゼロ（特徴量に効果なし）
- **p値**: H₀が正しいとして、この程度の差が偶然生じる確率
- **有意水準 α**: False Positive Rate の許容上限

### 機械学習の視点：モデル評価
- **交差検証**: 「このモデルは汎化性能があるか」を検証
- **t検定 ↔ 交差検証のt検定**: 2モデルの性能差の有意性検定
- **Type I/II Error ↔ Precision/Recall のトレードオフ**

In [ ]:
# 仮説検定とモデル評価の対応

# --- 1. 統計的仮説検定：係数の有意性 ---
from scipy import stats as scipy_stats

# OLS回帰の係数のt検定（手動計算）
n_stat = 150
X_stat = np.random.randn(n_stat, 3)
# β1=2.0（有意）、β2=0.1（小さい）、β3=0.0（無効）
true_betas = np.array([2.0, 0.1, 0.0])
y_stat = 1.0 + X_stat @ true_betas + np.random.randn(n_stat) * 1.5

X_stat_int = np.column_stack([np.ones(n_stat), X_stat])
beta_hat = np.linalg.inv(X_stat_int.T @ X_stat_int) @ X_stat_int.T @ y_stat

# 残差と標準誤差
y_hat = X_stat_int @ beta_hat
residuals_stat = y_stat - y_hat
s2 = np.sum(residuals_stat**2) / (n_stat - 4)  # s²
var_beta = s2 * np.linalg.inv(X_stat_int.T @ X_stat_int)
se_beta = np.sqrt(np.diag(var_beta))

# t統計量とp値
t_stats = beta_hat / se_beta
p_values = 2 * scipy_stats.t.sf(np.abs(t_stats), df=n_stat-4)

print('=== 統計：回帰係数のt検定 ===')
print(f'\n{'変数':<10} {'真の値':>8} {'推定値':>8} {'標準誤差':>8} {'t値':>8} {'p値':>10} {'有意性':>8}')
print('-' * 70)
var_names = ['切片', 'β₁(=2.0)', 'β₂(=0.1)', 'β₃(=0.0)']
true_vals = [1.0, 2.0, 0.1, 0.0]
for i, name in enumerate(var_names):
    sig = '***' if p_values[i] < 0.001 else ('**' if p_values[i] < 0.01 else ('*' if p_values[i] < 0.05 else 'n.s.'))
    print(f'{name:<10} {true_vals[i]:>8.2f} {beta_hat[i]:>8.4f} {se_beta[i]:>8.4f} {t_stats[i]:>8.3f} {p_values[i]:>10.4f} {sig:>8}')

In [ ]:
# --- 2. MLの交差検証 ↔ 統計的検定 ---
from sklearn.model_selection import cross_val_score

# モデルA（真に有効な特徴のみ）vs モデルB（ノイズ特徴含む）
X_useful = X_stat[:, [0]]      # β₁のみ（有効）
X_noisy = X_stat[:, [0, 2]]    # β₁ + β₃（ノイズ含む）
X_full = X_stat                # 全特徴

kf = KFold(n_splits=10, shuffle=True, random_state=42)

scores_A = cross_val_score(LinearRegression(), X_useful, y_stat, cv=kf,
                           scoring='neg_mean_squared_error')
scores_B = cross_val_score(LinearRegression(), X_noisy, y_stat, cv=kf,
                           scoring='neg_mean_squared_error')
scores_C = cross_val_score(LinearRegression(), X_full, y_stat, cv=kf,
                           scoring='neg_mean_squared_error')

# モデルAとCの差の有意性（対応のあるt検定）
t_stat, p_val = scipy_stats.ttest_rel(-scores_A, -scores_C)

print('=== ML：交差検証でのモデル比較 ===')
print(f'\nモデルA（有効特徴のみ）:  平均MSE = {-scores_A.mean():.4f} ± {scores_A.std():.4f}')
print(f'モデルB（ノイズ含む）:      平均MSE = {-scores_B.mean():.4f} ± {scores_B.std():.4f}')
print(f'モデルC（全特徴）:          平均MSE = {-scores_C.mean():.4f} ± {scores_C.std():.4f}')
print(f'\nモデルA vs C の対応t検定:')
print(f'  t値 = {t_stat:.4f}, p値 = {p_val:.4f}')
print(f'  → {'有意差あり（p<0.05）' if p_val < 0.05 else '有意差なし（p≥0.05）'}')

print('\n💡 ポイント:')
print('  統計のt検定 ≈ MLの交差検証 + 対応のあるt検定')
print('  「係数が有意か」≈「特徴量が予測に貢献するか」')

In [ ]:
# Type I/II Error と Precision/Recall の対応
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 仮説検定の判断マトリクス
matrix_data_stat = np.array([[0, 0], [0, 0]])
table_stat = [['', 'H₀真（効果なし）', 'H₁真（効果あり）'],
              ['棄却しない（有意でない）', '正しい判断\n（True Negative）', 'Type II Error（β）\n検出力不足'],
              ['棄却する（有意）', 'Type I Error（α）\nFalse Positive', '正しい判断\n（True Positive）']]

# テキストテーブルで表示
axes[0].axis('off')
colors = [['#f0f0f0', '#lightblue', '#lightblue'],
          ['#d4edda', '#d4edda', '#f8d7da'],
          ['#f8d7da', '#f8d7da', '#d4edda']]

col_labels = ['', 'H₀真\n（効果なし）', 'H₁真\n（効果あり）']
row_labels = ['', '判定:\n棄却しない', '判定:\n棄却する']
cell_data = [['', '✓ 正判断\n(TN)', '✗ Type II Error\n偽陰性（β）'],
             ['', '✗ Type I Error\n偽陽性（α）', '✓ 正判断\n(TP)']]

tbl = axes[0].table(
    cellText=cell_data,
    rowLabels=['棄却しない', '棄却する'],
    colLabels=['', 'H₀真（効果なし）', 'H₁真（効果あり）'],
    cellLoc='center',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 3)
axes[0].set_title('統計：仮説検定の判断マトリクス\n（Type I/II Error）', fontsize=11)

# MLの混同行列
cell_data_ml = [['', 'TN\n真陰性', 'FN\n偽陰性'],
                ['', 'FP\n偽陽性', 'TP\n真陽性']]

tbl2 = axes[1].table(
    cellText=cell_data_ml,
    rowLabels=['予測:陰性', '予測:陽性'],
    colLabels=['', '実際:陰性', '実際:陽性'],
    cellLoc='center',
    loc='center'
)
tbl2.auto_set_font_size(False)
tbl2.set_fontsize(9)
tbl2.scale(1, 3)
axes[1].axis('off')
axes[1].set_title('ML：混同行列\n（Precision/Recall）', fontsize=11)

plt.suptitle('統計のType I/II Error = MLのFPR/FNR（混同行列）', fontsize=12, y=1.05)
plt.tight_layout()
plt.show()

print('対応表:')
print('  統計: Type I Error（α）  ↔  ML: FPR (1-特異度) = 1 - Precision[陰性クラス]')
print('  統計: Type II Error（β） ↔  ML: FNR = 1 - Recall')
print('  統計: 検出力（1-β）      ↔  ML: Recall (= Sensitivity)')
print('  統計: 有意水準（α）      ↔  ML: FPR')

---
## 5. 情報量基準（AIC/BIC） → モデル選択・過学習防止

### 統計学：AIC/BIC
$$\text{AIC} = -2\ell(\hat{\boldsymbol{\theta}}) + 2k$$
$$\text{BIC} = -2\ell(\hat{\boldsymbol{\theta}}) + k\log n$$

ここで $k$ はパラメータ数、$\ell$ は最大対数尤度。

### 機械学習との対応
- $-2\ell$ = **訓練損失**（小さいほど良い）
- $2k$ or $k\log n$ = **複雑さペナルティ**（正則化と同じ思想）
- AIC/BIC最小化 ≈ **交差検証によるモデル選択**

In [ ]:
# AIC/BICと交差検証の比較
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

# 1次元の非線形データ
np.random.seed(42)
n_cv = 80
x_cv = np.linspace(-3, 3, n_cv)
y_cv = np.sin(x_cv) + np.random.randn(n_cv) * 0.3
X_cv = x_cv.reshape(-1, 1)

degrees = range(1, 12)
aics, bics, cv_scores_list = [], [], []

for deg in degrees:
    # 多項式特徴変換
    poly = PolynomialFeatures(degree=deg)
    X_poly = poly.fit_transform(X_cv)
    k = X_poly.shape[1]  # パラメータ数

    # OLS フィッティング
    lr_poly = LinearRegression(fit_intercept=False)
    lr_poly.fit(X_poly, y_cv)
    y_hat_poly = lr_poly.predict(X_poly)

    # 対数尤度（正規分布仮定）
    rss = np.sum((y_cv - y_hat_poly)**2)
    sigma2_hat = rss / n_cv
    log_lik = -n_cv/2 * np.log(2*np.pi*sigma2_hat) - rss/(2*sigma2_hat)

    # AIC/BIC
    aic = -2*log_lik + 2*k
    bic = -2*log_lik + k*np.log(n_cv)
    aics.append(aic)
    bics.append(bic)

    # 交差検証（10-fold）
    pipe = Pipeline([('poly', PolynomialFeatures(degree=deg)),
                     ('lr', LinearRegression())])
    cv_s = cross_val_score(pipe, X_cv, y_cv, cv=10,
                           scoring='neg_mean_squared_error')
    cv_scores_list.append(-cv_s.mean())

# 正規化してプロット
aics_norm = (np.array(aics) - min(aics)) / (max(aics) - min(aics))
bics_norm = (np.array(bics) - min(bics)) / (max(bics) - min(bics))
cv_norm = (np.array(cv_scores_list) - min(cv_scores_list)) / (max(cv_scores_list) - min(cv_scores_list))

best_aic = np.argmin(aics) + 1
best_bic = np.argmin(bics) + 1
best_cv = np.argmin(cv_scores_list) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# モデル選択基準の比較
axes[0].plot(list(degrees), aics_norm, 'b-o', label=f'AIC (最良: 次数{best_aic})', lw=2)
axes[0].plot(list(degrees), bics_norm, 'r-s', label=f'BIC (最良: 次数{best_bic})', lw=2)
axes[0].plot(list(degrees), cv_norm, 'g-^', label=f'10-fold CV (最良: 次数{best_cv})', lw=2)
axes[0].axvline(best_aic, color='b', linestyle='--', alpha=0.5)
axes[0].axvline(best_bic, color='r', linestyle='--', alpha=0.5)
axes[0].axvline(best_cv, color='g', linestyle='--', alpha=0.5)
axes[0].set_xlabel('多項式の次数')
axes[0].set_ylabel('正規化スコア（小さいほど良い）')
axes[0].set_title('AIC/BIC vs 交差検証\nモデル選択基準の比較')
axes[0].legend()

# 各次数のフィッティング曲線
x_plot = np.linspace(-3, 3, 200).reshape(-1, 1)
colors_deg = ['blue', 'green', 'red']
selected_degrees = [best_bic, best_aic, 10]
labels_deg = [f'次数{best_bic}（BIC最良）', f'次数{best_aic}（AIC最良）', '次数10（過学習）']

axes[1].scatter(x_cv, y_cv, color='gray', alpha=0.5, s=20, label='データ')
x_true = np.linspace(-3, 3, 200)
axes[1].plot(x_true, np.sin(x_true), 'k-', lw=2, label='真の関数 sin(x)')

for deg, color, label in zip(selected_degrees, colors_deg, labels_deg):
    pipe = Pipeline([('poly', PolynomialFeatures(degree=deg)),
                     ('lr', LinearRegression())])
    pipe.fit(X_cv, y_cv)
    y_plot = pipe.predict(x_plot)
    axes[1].plot(x_plot, y_plot, color=color, lw=2, label=label, alpha=0.8)

axes[1].set_ylim(-2, 2)
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].set_title('多項式次数と過学習')
axes[1].legend(fontsize=9)

plt.suptitle('情報量基準（AIC/BIC） ≈ 交差検証：過学習防止の統計的根拠', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f'最適次数: AIC={best_aic}, BIC={best_bic}, 10-fold CV={best_cv}')
print('💡 BICはAICよりペナルティが大きく、よりシンプルなモデルを選びやすい')

---
## 6. まとめ：統計的視点でMLを読む

### 完全な対応表

In [ ]:
# 対応表の表示
summary_data = {
    '統計学の概念': [
        '線形回帰（OLS）',
        '最尤推定（MLE）',
        '正規分布仮定',
        'ラプラス分布仮定',
        'ベイズ推定（MAP）＋正規事前分布',
        'ベイズ推定（MAP）＋ラプラス事前分布',
        '有意水準（α）',
        '検出力（1-β）',
        '残差の正規性確認',
        'AIC（赤池情報量基準）',
        'BIC（ベイズ情報量基準）',
        '交差検証（統計）',
    ],
    '機械学習の概念': [
        'LinearRegression（MSE損失最小化）',
        '損失関数最小化',
        'MSE（二乗誤差損失）',
        'MAE（絶対誤差損失）',
        'Ridge回帰（L2正則化）',
        'LASSO（L1正則化）',
        'FPR（偽陽性率）のしきい値',
        'Recall（再現率）',
        '残差プロット・QQプロット',
        '正則化＋訓練誤差のバランス',
        'より強い正則化（BIC > AIC）',
        '交差検証（CV）',
    ]
}

df_summary = pd.DataFrame(summary_data)
print('=== 統計学 ↔ 機械学習 対応表 ===')
print(df_summary.to_string(index=False))

In [ ]:
# 最終まとめ：総合デモ
print('=' * 60)
print('  統計学 → 機械学習：橋渡しのまとめ')
print('=' * 60)
print('''
【根本的なつながり】

1. OLS = MSE損失最小化
   「最小二乗法」と「勾配降下法」は同じ目標を異なる方法で解く

2. MLE = 損失関数選択の根拠
   分布仮定が損失関数を決める：
   正規分布 → MSE、ラプラス → MAE、ベルヌーイ → 交差エントロピー

3. MAP推定 = 正則化
   事前分布が正則化の種類を決める：
   正規事前 → Ridge（L2）、ラプラス事前 → LASSO（L1）

4. 仮説検定 = モデル評価
   「係数が有意か」= 「特徴量が汎化に貢献するか」
   Type I Error = FPR、Type II Error = FNR、検出力 = Recall

5. AIC/BIC = 過学習防止
   パラメータペナルティ = 正則化の思想
   情報量基準 ≈ 交差検証によるモデル選択

【実践へのヒント】
- MLで「なぜRidgeを使うか」= 「係数に正規事前分布を仮定するから」
- MLで「なぜ交差検証するか」= 「AIC/BICと同様に過学習を防ぐため」
- MLで「損失関数を何にするか」= 「残差の分布に何を仮定するかで決まる」
''')
print('=' * 60)

---
## 参考文献

- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer.
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning*. Springer.
- 統計数理研究所. 統計検定準1級公式テキスト.
- statspg/notebooks: 01〜32（本シリーズの前ノートブック）
- mlpg/notebooks/fundamentals: 04_linear_models, 10_hyperparameter_tuning